# LangChain Tool Calling — Python Notebook (Groq Free Tier)

Uses [Groq](https://console.groq.com) — free API, no credit card required.  
**No LangGraph dependency** — pure `langchain-core` + `langchain-groq`.

## 0. Setup

In [ ]:
# pip install langchain-core langchain-groq
import os
os.environ["GROQ_API_KEY"] 

MODEL = "llama-3.1-8b-instant"

---
## 1. Define Tools

The `@tool` decorator wraps a Python function into a LangChain tool.  
The **docstring** is what the LLM reads to decide whether to call this tool.

In [2]:
from langchain_core.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a location."""
    if location.lower() in ["sf", "san francisco"]:
        return "It's 60 degrees and foggy."
    return "It's 90 degrees and sunny."

@tool
def get_coolest_cities() -> str:
    """Get a list of the coolest cities."""
    return "nyc, sf"

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

tools = [get_weather, get_coolest_cities, multiply]
tools_by_name = {t.name: t for t in tools}

---
## 2. Call a Tool Directly (no LLM)

Tools are just Python functions — you can call them directly with `.invoke()`.

In [3]:
# Plain invocation
print(multiply.invoke({"a": 6, "b": 7}))        # 42
print(get_weather.invoke({"location": "sf"}))    # It's 60 degrees and foggy.

42
It's 60 degrees and foggy.


In [4]:
# Invoke with a tool_call dict (what the LLM produces) → returns a ToolMessage
tool_call = {
    "name": "get_weather",
    "args": {"location": "sf"},
    "id": "tool_call_id",
    "type": "tool_call",
}
result = get_weather.invoke(tool_call)
print(result)
# ToolMessage(content="It's 60 degrees and foggy.", name='get_weather', tool_call_id='tool_call_id')

content="It's 60 degrees and foggy." name='get_weather' tool_call_id='tool_call_id'


In [5]:
# Multiple tool calls in parallel — call each tool directly
tool_calls = [
    {"name": "get_coolest_cities", "args": {},                  "id": "id_1", "type": "tool_call"},
    {"name": "get_weather",        "args": {"location": "sf"}, "id": "id_2", "type": "tool_call"},
]

results = [tools_by_name[tc["name"]].invoke(tc) for tc in tool_calls]
for r in results:
    print(r)

content='nyc, sf' name='get_coolest_cities' tool_call_id='id_1'
content="It's 60 degrees and foggy." name='get_weather' tool_call_id='id_2'


---
## 3. Attach Tools to LLM (`bind_tools`)

`bind_tools()` sends the tool JSON schemas to the LLM so it knows the tools exist and how to call them.

In [6]:
from langchain_groq import ChatGroq

llm = ChatGroq(model=MODEL)
model_with_tools = llm.bind_tools(tools)

# LLM response when a tool call is needed
response = model_with_tools.invoke("what's the weather in sf?")

print("RAW RESPONSE:\n", response)
print("\nCONTENT:\n", response.content)
print("\nTOOL CALLS:\n", response.tool_calls)
# [{'name': 'get_weather', 'args': {'location': 'sf'}, 'id': '...', 'type': 'tool_call'}]

RAW RESPONSE:
 content='' additional_kwargs={'tool_calls': [{'id': 'yys3gfr7e', 'function': {'arguments': '{"location":"sf"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 322, 'total_tokens': 336, 'completion_time': 0.028498698, 'completion_tokens_details': None, 'prompt_time': 0.020593699, 'prompt_tokens_details': None, 'queue_time': 0.019734423, 'total_time': 0.049092397}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_6a1eabf260', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e9ebd-e2c0-7de2-abcc-b2978615e39f-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'sf'}, 'id': 'yys3gfr7e', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 322, 'output_tokens': 14, 'total_tokens': 336}

CONTENT:
 

TOOL CALLS:
 [{'name': 'get_weather', 'args': {'location': 'sf'}, 'id': 'yys3gfr7e', 'type'

In [7]:
# Execute the tool the LLM requested
tool_call = response.tool_calls[0]
tool_result = tools_by_name[tool_call["name"]].invoke(tool_call)
print(tool_result)
# ToolMessage(content="It's 60 degrees and foggy.", ...)

content="It's 60 degrees and foggy." name='get_weather' tool_call_id='yys3gfr7e'


---
## 4. Full Tool-Calling Loop (from scratch)

Manual agentic loop — no graph, no nodes:

```
User query
  → LLM decides: call a tool?
  → if yes  → execute tool → feed result back → LLM again
  → if no   → return final answer
```

In [8]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
import json

def run_agent(user_input: str, max_steps: int = 6, verbose: bool = True):
    messages = [HumanMessage(content=user_input)]

    for step in range(max_steps):

        # 1. LLM THINK + (maybe) TOOL CALL
        response = model_with_tools.invoke(messages)
        messages.append(response)

        tool_calls = getattr(response, "tool_calls", None)

        # 2. TERMINATION: no tool call => final answer
        if not tool_calls:
            return messages

        # 3. STRICT: only allow ONE reasoning cycle at a time
        if len(tool_calls) > 1:
            raise ValueError("Multiple tool calls in one step not allowed in strict ReAct")

        tc = tool_calls[0]
        name = tc["name"]
        args = tc.get("args", {})

        # 4. HARD GUARD: block Groq-breaking nested tool syntax
        for v in args.values():
            if "<function=" in str(v) or "function=" in str(v):
                raise ValueError(f"Blocked nested tool call: {args}")

        # 5. EXECUTE TOOL
        tool_result = tools_by_name[name].invoke(args)

        # 6. FLATTEN TOOL OUTPUT (CRITICAL FOR GROQ STABILITY)
        if isinstance(tool_result, dict) or isinstance(tool_result, list):
            content = json.dumps(tool_result)
        else:
            content = str(tool_result)

        # 7. STRIP ANY FUNCTION-LIKE TOKENS (VERY IMPORTANT)
        content = (
            content
            .replace("<function=", "")
            .replace("</function>", "")
            .replace("function=", "")
        )

        tool_msg = ToolMessage(
            content=content,
            tool_call_id=tc.get("id")
        )

        messages.append(tool_msg)

        if verbose:
            print(f"[step {step}] TOOL → {name}({args}) = {content}")

    raise RuntimeError("Max steps reached without final answer")

# --- Run ---
messages = run_agent("what's the weather in sf?")
print("\nFinal answer:", messages[-1].content)

[step 0] TOOL → get_weather({'location': 'sf'}) = It's 60 degrees and foggy.
[step 1] TOOL → get_weather({'location': 'san francisco'}) = It's 60 degrees and foggy.
[step 2] TOOL → get_weather({'location': 'san francisco, ca'}) = It's 90 degrees and sunny.

Final answer: The weather forecast in San Francisco, CA is quite variable. It can be foggy in the morning and then sunny in the afternoon.


In [9]:
# Print full message trace
for msg in messages:
    print(f"{type(msg).__name__:15} | {str(msg.content)[:100]}")

HumanMessage    | what's the weather in sf?
AIMessage       | 
ToolMessage     | It's 60 degrees and foggy.
AIMessage       | 
ToolMessage     | It's 60 degrees and foggy.
AIMessage       | 
ToolMessage     | It's 90 degrees and sunny.
AIMessage       | The weather forecast in San Francisco, CA is quite variable. It can be foggy in the morning and then


In [10]:
# Multi-step: requires two tool calls chained together
messages = run_agent("what's the weather in the coolest cities?")
print("\nFinal answer:", messages[-1].content)

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=get_weather>{"location": "get_coolest_cities"</function>\n\n'}}

In [11]:
# Arithmetic: no tool call needed path
messages = run_agent("what's 42 x 7?")
print("\nFinal answer:", messages[-1].content)

[step 0] TOOL → multiply({'a': 42, 'b': 7}) = 294

Final answer: The result of the function call is 294.


---
## 5. Tool Customization

In [12]:
# Parse docstring for per-parameter descriptions
@tool("multiply_tool", parse_docstring=True)
def multiply_v2(a: int, b: int) -> int:
    """Multiply two numbers.

    Args:
        a: First operand
        b: Second operand
    """
    return a * b

print(multiply_v2.name)
print(multiply_v2.description)
print(multiply_v2.args)

multiply_tool
Multiply two numbers.
{'a': {'description': 'First operand', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second operand', 'title': 'B', 'type': 'integer'}}


In [13]:
# Explicit Pydantic schema
from pydantic import BaseModel, Field

class MultiplyInputSchema(BaseModel):
    """Multiply two numbers"""
    a: int = Field(description="First operand")
    b: int = Field(description="Second operand")

@tool("multiply_tool", args_schema=MultiplyInputSchema)
def multiply_v3(a: int, b: int) -> int:
    return a * b

print(multiply_v3.args)

{'a': {'description': 'First operand', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second operand', 'title': 'B', 'type': 'integer'}}


---
## 6. Error Handling

In [18]:
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage

@tool
def risky_multiply(a: int, b: int) -> int:
    """Multiply two numbers. Raises if a is 42."""
    if a == 42:
        raise ValueError("Cannot use 42 as first operand")
    return a * b

# Call directly — exception propagates normally
try:
    risky_multiply.invoke({"a": 42, "b": 7})
except ValueError as e:
    print("Caught:", e)

# Call with tool_call dict — error is captured in a ToolMessage (status='error')
tool_call = {
    "name": "risky_multiply",
    "args": {"a": 42, "b": 7},
    "id": "id_1",
    "type": "tool_call",
}

try:
    result = risky_multiply.invoke(tool_call["args"])
    content = str(result)
    status = "success"
except Exception as e:
    content = str(e)
    status = "error"

tool_msg = ToolMessage(
    content=content,
    tool_call_id=tool_call["id"]
)

print("status:", status)
print("content:", tool_msg.content)

Caught: Cannot use 42 as first operand
status: error
content: Cannot use 42 as first operand


---
## Summary: The Tool-Calling Loop

```
1. model.bind_tools(tools)       → sends tool JSON schemas to LLM
2. LLM outputs AIMessage         → contains tool_calls=[{name, args}] if it wants a tool
3. tool.invoke(tool_call)        → executes the Python function, returns ToolMessage
4. ToolMessage appended          → fed back into message history
5. LLM invoked again             → sees result, produces final answer
6. No tool_calls in response     → loop ends
```

| Object | Role |
|---|---|
| `model.bind_tools()` | Registers tool schemas with the LLM |
| `AIMessage.tool_calls` | LLM's decision to invoke a tool |
| `tool.invoke(tool_call)` | Executes the actual Python function |
| `ToolMessage` | Tool result fed back into context |
| `tools_by_name` | Dict for dispatching calls by name |
| `while not response.tool_calls` | Routing logic that closes the loop |